# Notebook 02 — Source and Text Similarity Grouping (Experiment 2)

**Research context:** Component 4 — AI-Driven Governance, Compliance and Trust Infrastructure  
**Research question:** Do records from different audit sources share near-identical narrative templates that could inflate cross-source cross-validation scores, and what is the effect of accounting for this when evaluating the TF-IDF + LR baseline?

**Experiment 2** extends Experiment 1 by combining source-group identity with text-similarity grouping.  
Records with **Jaccard similarity ≥ 0.50** on five-word shingles are merged into the same cross-validation group, regardless of source.  
This is a **conservative sensitivity experiment** — it cannot prove complete leakage removal or establish incident identity.

**Caveat:** All results are exploratory development evidence. No unseen-data generalization is implied.

---

## How to use in Colab
1. Upload and unzip the repository ZIP.  
2. Set `ML_ROOT` in the Setup cell to the extracted ML directory path.  
3. Run all cells — no models are trained, no data is modified.

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import sys
from pathlib import Path

ML_ROOT = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.').resolve().parent
# ML_ROOT = Path("/content/project/.../ML")  # Colab: set this manually

SRC_DIR = ML_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"ML_ROOT: {ML_ROOT}")
print(f"ML_ROOT exists: {ML_ROOT.exists()}")

In [ ]:
# Optional installation (uncomment if needed):
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ML_ROOT / "requirements.txt")], check=True)

In [ ]:
# =============================================================================
# IMPORTS AND VERSIONS
# =============================================================================
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

print(f"Python:      {sys.version.split()[0]}")
print(f"NumPy:       {np.__version__}")
print(f"Pandas:      {pd.__version__}")
print(f"scikit-learn:{sklearn.__version__}")
print(f"Matplotlib:  {matplotlib.__version__}")

In [ ]:
# =============================================================================
# ARTIFACT PATHS AND DEPENDENCY CHECK
# =============================================================================
DATA_CSV = ML_ROOT / "data" / "state_land_governance_confirmed_200.csv"
EXP2_DIR = ML_ROOT / "results" / "experiment2_source_text"

MANIFEST_CSV      = EXP2_DIR / "grouping_manifest.csv"
SUMMARY_JSON      = EXP2_DIR / "grouping_summary.json"
SIMILARITY_CSV    = EXP2_DIR / "similarity_pairs.csv"
OOF_CSV           = EXP2_DIR / "out_of_fold_predictions.csv"
METRICS_JSON      = EXP2_DIR / "experiment2_metrics.json"
CM_CSV            = EXP2_DIR / "confusion_matrix.csv"

required = {
    "dataset": DATA_CSV, "grouping_manifest": MANIFEST_CSV,
    "grouping_summary": SUMMARY_JSON, "similarity_pairs": SIMILARITY_CSV,
    "oof_predictions": OOF_CSV, "metrics_json": METRICS_JSON,
}
for name, path in required.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"  {status}: {name} -> {path.name}")

missing = [p for p in required.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} required artifact(s) missing. "
        "Ensure the Experiment 2 results directory is present."
    )

# Dataset identity
dataset_hash = hashlib.sha256(DATA_CSV.read_bytes()).hexdigest()
EXPECTED_HASH = "26979f18f292f8a33a5ff960123a168f0652a5d8e11032b930b4dc7fcf070c61"
print(f"\nDataset hash match: {dataset_hash == EXPECTED_HASH}")

## 1. Source Groups and Text-Similarity Grouping

In [ ]:
# Load grouping manifest and verify weighted total = 200
manifest = pd.read_csv(MANIFEST_CSV)
print(f"Manifest rows: {len(manifest)} (should equal 200)")
assert len(manifest) == 200, "Manifest row count does not equal 200"

# Group-size distribution
group_sizes = manifest.groupby("Combined_Group").size()
size_dist = group_sizes.value_counts().sort_index()
weighted_total = (size_dist.index * size_dist.values).sum()
print(f"Weighted total (group_size × count): {weighted_total} (should equal {len(manifest)})")
assert weighted_total == len(manifest), "Weighted total mismatch"

print("\nGroup-size distribution:")
for sz, cnt in size_dist.items():
    print(f"  Groups of size {int(sz):>3}: {int(cnt):>4} group(s) — {int(sz)*int(cnt):>4} records")
print(f"  Total groups: {group_sizes.nunique()}")

In [ ]:
# Grouping summary JSON
with open(SUMMARY_JSON, "r") as f:
    summary = json.load(f)
print("Grouping summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

In [ ]:
# Similarity pairs
sim_df = pd.read_csv(SIMILARITY_CSV)
print(f"Qualifying similarity pairs (Jaccard >= 0.50): {len(sim_df)}")
if len(sim_df) > 0:
    print(sim_df[["ID_A", "ID_B", "Jaccard"]].describe())
    print(f"\nSample of qualifying pairs:")
    print(sim_df[["ID_A", "ID_B", "Jaccard"]].head(10).to_string(index=False))

In [ ]:
# Largest combined group
largest_group = group_sizes.idxmax()
largest_size  = group_sizes.max()
largest_members = manifest[manifest["Combined_Group"] == largest_group]
print(f"Largest group: {largest_group} with {largest_size} records")
print("\nLabel distribution within largest group:")
print(largest_members["ML_Label_4Class"].value_counts().to_string())
print(
    "\nNote: This group's size reflects shared narrative boilerplate, not a claim that all "
    "members describe the same incident or that grouping eliminates all leakage."
)

## 2. Fold Sizes and Class Distributions

In [ ]:
oof = pd.read_csv(OOF_CSV)
print(f"OOF records: {len(oof)}, unique IDs: {oof['Research_ID'].nunique()}")

LABEL_ORDER = [
    "Administrative / Procedural / Integrity",
    "Lease Revenue / Payment / Enforcement",
    "Unauthorized Allocation / Transfer / Use",
    "Protected / Environmental Lease Misuse",
]

# Fold sizes and label distribution
fold_summary = oof.groupby("Fold").apply(
    lambda g: pd.Series({
        "n": len(g),
        "accuracy": round((g["True_Label"] == g["Predicted_Label"]).mean(), 4),
        **{lbl[:20]: int((g["True_Label"]==lbl).sum()) for lbl in LABEL_ORDER}
    })
).reset_index()
print("\nPer-fold test-set sizes and label counts:")
print(fold_summary.to_string(index=False))

## 3. Experiment 2 Metrics

In [ ]:
y_true = oof["True_Label"].tolist()
y_pred = oof["Predicted_Label"].tolist()

acc      = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro",    labels=LABEL_ORDER, zero_division=0)
wt_f1    = f1_score(y_true, y_pred, average="weighted", labels=LABEL_ORDER, zero_division=0)
macro_p  = precision_score(y_true, y_pred, average="macro", labels=LABEL_ORDER, zero_division=0)
macro_r  = recall_score(y_true, y_pred, average="macro",    labels=LABEL_ORDER, zero_division=0)
n_wrong  = int(sum(a != b for a, b in zip(y_true, y_pred)))

print("Experiment 2 — Overall OOF Metrics (source+text-grouped CV, n=200)")
print(f"  Accuracy:           {acc:.4f}")
print(f"  Macro-Precision:    {macro_p:.4f}")
print(f"  Macro-Recall:       {macro_r:.4f}")
print(f"  Macro-F1:           {macro_f1:.4f}")
print(f"  Weighted-F1:        {wt_f1:.4f}")
print(f"  Misclassified:      {n_wrong} / {len(y_true)}")

In [ ]:
# Per-class metrics
cm = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
pc_rows = []
for i, lbl in enumerate(LABEL_ORDER):
    tp = cm[i,i]; fp = int(cm[:,i].sum()-tp); fn = int(cm[i,:].sum()-tp)
    tn = int(cm.sum()-tp-fp-fn)
    p  = tp/(tp+fp) if tp+fp>0 else 0
    r  = tp/(tp+fn) if tp+fn>0 else 0
    f1 = 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn)>0 else 0
    fpr= fp/(fp+tn) if (fp+tn)>0 else 0
    pc_rows.append({"Class": lbl[:35], "Precision": round(p,4), "Recall": round(r,4),
                    "F1": round(f1,4), "FPR": round(fpr,4), "Support": tp+fn})
pc_df = pd.DataFrame(pc_rows).set_index("Class")
print("Per-class metrics (Experiment 2):")
print(pc_df.to_string())
print("\nNote: F1 values are per-class F1 scores, not macro-averaged.")

In [ ]:
# Confusion matrix
short_labels = [l.split("/")[0].strip()[:20] for l in LABEL_ORDER]
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=short_labels)
fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(ax=ax, colorbar=True, cmap="Blues")
ax.set_title("Experiment 2 — Confusion Matrix\n(Source+Text-Grouped CV, OOF Predictions, n=200)")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
plt.tight_layout()
plt.show()

## 4. Limitations

- **Text similarity grouping does not prove incident identity.** Two records sharing similar phrasing may describe different incidents with similar reporting templates. Grouping them together is a conservative protective measure, not a factual claim.
- **Complete leakage removal is not established.** There may be subtler forms of leakage (shared vocabulary, regional terminology, category-specific phrasing) that grouping by source or shingle similarity does not address.
- **The largest group (26 records) has a disproportionate effect on Fold 3.** All 26 records test together in one fold, making Fold 3 results more sensitive to this group's characteristics than to the general distribution.
- Results are development cross-validation evidence. No generalization claim is made.